# 🔄 Reinforcement Learning!

📊 **High-level goal:**
Train an agent (the “learner”) to interact with the given environment and learn from it, so it can maximize its cumulative rewards over time.
*   **Environment:**
     *   The environment **is already created** and set up for you with the help of the
"gymnasium" library.
     *   The agent interacts with a simulated environment, which has states, accepts actions, and returns rewards. The environment resets at the beginning of each episode, and the agent takes actions within it, receiving feedback in the form of rewards.
*   **Q-table:**
     *   This table stores information about the "quality" of actions in different states.
     *   The goal of the agent is to update the Q-table over time to better estimate the value of taking different actions in various states.
     *   You will initialize and learn a Q-table in this project.

## 🗂️ Project structure

1. Following the task description, you'll find a set of formal requirements that your solution must meet.
2. After completing each task, you must answer a series of questions, marked with ❓, to demonstrate your understanding of the techniques used.

## 📝 Submission guidelines

- For each problem, provide your code solution and model results inside this notebook.
- Answer the follow-up questions in markdown format within this notebook. A few sentences are enough; there is no length requirement.
- Ensure your explanations are clear, concise, and demonstrate a deep understanding of the techniques employed.
- Submit this notebook to Canvas. Deliver the notebook with its outputs saved; ideally, *restart* the kernel and then *run all*.

Good luck!

---

### 📮 Imports:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from utils import *

# note: make sure to install necessary libraries like imageio, gymnasium etc.
# !pip install gymnasium

In [ ]:
import gymnasium as gym
from gymnasium.spaces import Discrete, MultiDiscrete

### Environment class with necessary functions - provided for you:

In [ ]:
class TrondheimEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, conf: dict):
        """
        Every environment should be derived from gym.Env and at least contain
        variables observation_space and action_space specifying the type of possible
        observations and actions using spaces.Box or spaces.Discrete.

        Example:
        >>> EnvTest = TrondheimEnv()
        >>> EnvTest.observation_space=spaces.Box(low=-1, high=1, shape=(3,4))
        >>> EnvTest.action_space=spaces.Discrete(2)
        """

        self.verbose = True  # Flag to control printouts

        # action definition:
        NUM_ACTIONS = 4
        self.action_space = Discrete(NUM_ACTIONS)

        self.env_image_path = conf['env_image_path']

        self.pallete = np.array([
                            [0, 0, 255],    # Blue:0:Water
                            [255, 255, 0],  # Yellow:1:Start
                            [0, 255, 0],    # Green:2:Land
                            [255, 0, 0],    # Red:3:BusLane
                            [128, 0, 128],  # Purple:4:BusStop
                            [0, 0, 0]],      # Black:5:Final Location
                           dtype=np.uint8)
                            # [128, 128, 0]], # Olive:5:Final Location

        self.neighbours = np.array([[-1,0],[0,-1],[1,0],[0,1]]) # up, left, down, right
        self.not_allowed_area = [0] # water
        self.start_class = 1 # start
        self.normal_land_class = 2 # land
        self.busstop_class = 4 # bus stop
        self.final_location_class = 5 # final location (2)

        # does the agent die when entering not_allowed_area
        self.dead_allowed = conf['dead_allowed']
        # should we start at different place each time
        self.start_random = conf['start_random']

        # read RGB image into a 3D numpy array
        img_3d = read_png_file(self.env_image_path, printDebug = False)

        # convert RGB image into 2D semantic class representation
        img_2d = make_2d(map_3d=img_3d, palette_array=self.pallete)

        # the current state of the environment
        self.observation = img_2d # initial observation - 2D semantic class
        # copy of the environment's initial state before any actions have been taken
        self.initial_observation = img_2d.copy()

        max_observation_value = np.max(img_2d)
        # observation space is a gym object (MultiDiscrete), like numpy array. It can be sampled
        observation_space = np.full_like(img_2d, fill_value=max_observation_value)
        self.observation_space = MultiDiscrete(observation_space)

        self.reward = 0
        self.terminated = False
        
    def step(self, action: int) -> tuple[np.ndarray, float, bool, dict]:
        """
        This method is the primary interface between environment and agent.

        Parameters:
            action: the index of the respective action

        Returns:
            output: information provided by the environment about its current state:
                    (observation, reward, terminated, info)

                    self.observation: The updated state of the environment (2D)
                    reward: Reward agent received for the action in this step 
                    terminated: indicate if the episode has ended
                    info: Additional information (not used, always None)

        """
        obs = self.observation # 2D array of semantic class image representation
        terminated = False
        info = None
        
        # just start and see where it goes
        start_idx = np.argwhere(obs == self.start_class)[0]
        # action is: 0:up, 1:left, 2:down, 3:right
        next_idx = start_idx + self.neighbours[action]
        reward = self.reward
        
        if self.verbose:
          print('Next step towards: ')

        if not is_valid_idx(image_2D=obs, selected_idx=next_idx):
            next_idx = start_idx
            if self.verbose:
                print('Hey there, you cannot escape outside of the environment')

        # now decide what happens next
        if obs[tuple(next_idx)] == self.normal_land_class:
            if self.verbose:
                print('Green Land class area')

        # check if it is a non allowed area
        if obs[tuple(next_idx)] in self.not_allowed_area:
            if self.verbose:
                print('The move is not allowed')
            if self.dead_allowed:
                # penalize the agent if allowed to die. Terminate
                reward = -1
                terminated = True
            else:
                # if NOT allowed to die, reset the next position to current one - prevent the move
                next_idx = start_idx

        # if the player is at a bus stop
        elif obs[tuple(next_idx)] == self.busstop_class:
            if self.verbose:
                print('A bus stop')

            # get the bus stop that is not the next step
            bus_stop_indices = np.argwhere(obs == self.busstop_class)
            # flip the next state to the other side of the stop
            if np.array_equal(next_idx, bus_stop_indices[0]):
                # teleport the agent from bus stop 0 to bus stop 1
                next_idx = bus_stop_indices[1]
            else:
                # teleport the agent from bus stop 1 to bus stop 0
                next_idx = bus_stop_indices[0]

        # and the Final Location
        elif obs[tuple(next_idx)] == self.final_location_class:
            if self.verbose:
                print('Final Location - well done!')
                print('You have discovered the Final Location of AI center in Trondheim')
            reward = 10
            terminated = True

        # now we need to change the observation space
        # i.e. update the environment’s state to reflect the agent's movement

        # if agent’s current position was initial start position
        if self.initial_observation[tuple(start_idx)] == self.start_class:
            # if true, update observation - mark the start position as neutral area
            self.observation[tuple(start_idx)] = self.normal_land_class
        else:
            # if current position not the initial start one
            # restores position in self.observation to its original value from self.initial_observation
            # re-color the observation as where you were if not start position
            self.observation[tuple(start_idx)] = self.initial_observation[tuple(start_idx)]

        # sets the new position of the agent as the next start position
        self.observation[tuple(next_idx)] = self.start_class

        self.terminated = self.terminated or terminated

        return self.observation, reward, terminated, info


    def reset(self) -> np.ndarray:
        """
        This method resets the environment to its initial values.
        """

        img_3d = read_png_file(self.env_image_path, printDebug = False)
        img_2d = make_2d(map_3d=img_3d, palette_array=self.pallete)

        if self.start_random:
          # we need to move the start to another place
          img_2d = swap_values(arr=img_2d, value1=1, value2=2)

        self.observation = img_2d
        self.initial_observation = img_2d.copy()

        max_observation_value = np.max(img_2d)
        observation_space = np.full_like(img_2d, fill_value=max_observation_value)
        self.observation_space = MultiDiscrete(observation_space)

        self.reward = 0
        self.terminated = False

        return self.observation
        
    def get_state_id(self):
        """
        This method converts the observation to a unique id that we will use to index the Q table.
        """
        rows, cols = self.observation.shape
        grid_size = rows * cols

        agent_pos = np.argwhere( self.observation == self.start_class)[0]
        agent_row, agent_col = agent_pos[0], agent_pos[1]

        grid_index = (agent_row * cols) + agent_col

        return grid_index

    def render(self, mode: str = "", close: bool = False) -> np.ndarray:
        """
        This method provides the option to render the environment's behavior to a
        window which should be readable to the human eye if mode is set to 'human'.
        """
        render_ready_arr = make_3d(map_2d=self.observation, palette_array=self.pallete)
        if mode == "human":
            plt.imshow(render_ready_arr)
            plt.show()
            plt.close()
        return render_ready_arr

    def close(self):
        img_3d = read_png_file(self.env_image_path)
        img_2d = make_2d(map_3d=img_3d, palette_array=self.pallete)
        self.observation = img_2d
        self.initial_observation = img_2d.copy()


### Configuring and loading the environment:

#### 🔎📉  Let's first visualize our simple toy environment

In [ ]:
# This is how you create a configuration for the environment
env_config = {
    'env_image_path':'image_simple.png',
    'dead_allowed':False,
    'start_random':False,
}


if not os.path.exists(env_config['env_image_path']):
  # make sure you have the folder and the image
  print(f"Error: The image path {env_config['env_image_path']} does not exist.")
else:
  # if everything fine, we create the environment
  env = TrondheimEnv(conf=env_config)
  print(f"Environment successfully created")

In [ ]:
# visualize the environment. Yellow is where we are, Black is where we want to go
env.reset()
_ = env.render("human")

Let's see how actions work:

In [ ]:
action_dict = {
    'up':0,
    'left':1,
    'down':2,
    'right':3,
}

In [ ]:
# select your action here
# you are encouraged to play further with this
my_action = 'left'

# now take a step through the environment by taking this action
obs, rew, term, info = env.step(action_dict[my_action])
print(f"Reward for action {my_action} is: {rew}")
env.render("human")

print(f"Observation space looks like this:\n{obs}")

## 🐮 Problem 1: Q-learning in a simple toy environment
### ❗️ The problem
In the first problem, you will train an agent to navigate from a start position (yellow) to a goal position (black) in a simple grid environment using Q-learning.

### 📋 Formal requirements
1. **Implementation**:
   - Initialize the Q-table
   - Write the necessary helper functions for Q-learning (exploration rate, learning rate and action selection)
   - Choose and tune the hyperparameters
   - Train the agent using Q-learning
   - Run one episode with the trained agent and verify that it reaches the goal
   - Visualize the learned Q-values as heatmaps
2. **Discussion**:
   - Why do we need an exploration rate?
   - Why do we update the learning rate?
   - Explain all terms of the Q-learning update rule.
   - What do the heatmaps show?
   - Why does the Q-table have 144 states?
   - What happens with γ = 0 or γ = 1?
   - Every step gives reward 0. How does the agent still learn the shortest path?
   - What happens if the exploration rate is 0 from the start?
   - Why do we set the exploration rate to 0 when we show the trained agent?

### 🐣 Let's learn how to do this in a simple environment

In [ ]:
env_config = {
    'env_image_path':'image_simple.png',
    'dead_allowed':False,
    'start_random':False,
}

if not os.path.exists(env_config['env_image_path']):
  print(f"Error: The image path {env_config['env_image_path']} does not exist.")
else:
  env = TrondheimEnv(conf=env_config)
  print(f"Environment successfully created")

### Step 1: Initialize Q table

In [ ]:
# Task 1: Initialize Q table
"""
TODO: given the environment, initialize the Q table
"""

# Tip: pay attention to the dimensions of Q table
# Tip: how many states does the table have, how many actions

# ====================================
# YOUR CODE GOES HERE
# ====================================

In [ ]:
# take a look at the initial Q table
print(f"Q table has this shape: {q_table.shape}\n")

# you may want to print Q table, or parts of it, to get the vibe
# print(q_table[:10])
print(f"\n")
# print(q_table[135:])

### ❓ Why does the Q-table have 144 states?

### Step 2: Write necessary helper functions for Q-learning

In [ ]:
def update_explore_rate(episode: int, epsilon_min: float, max_episodes: int):
    """controls the balance between exploration and exploitation"""
    # ====================================
    # YOUR CODE GOES HERE: return updated explore_rate
    # ====================================
    pass

def update_learning_rate(episode: int, min_learning_rate: float, max_episodes: int):
    """controls how aggressively the agent updates its knowledge"""
    # ====================================
    # YOUR CODE GOES HERE: return updated learning_rate
    # ====================================
    pass

def update_action(env, state: int, explore_rate: float, q_table: np.ndarray) -> int:
    """uses the exploration rate to decide whether to take a random action (explore) 
    or to act according to the learned Q-values (exploit)""" 
    # ====================================
    # YOUR CODE GOES HERE: return an action
    # ====================================
    pass


### ❓ Why do we need an exploration rate?


### ❓ Why do we update the learning rate?


### Step 3: Play with Hyperparameters

In [ ]:
# ====================================
# YOUR CODE GOES HERE: replace None with your values
# ====================================
GAMMA = None
NUM_EPISODES = None
MAX_STEPS_PER_EPISODE = None
EPSILON = None
EPSILON_MIN = None
MIN_LEARNING_RATE = None

### 🐤 Now use Q learning to learn the Q table
### Step 4: Train the agent using Q-learning

The Q-value update equation, derived from Bellman equation is as follows:


$$
Q(s, a) \gets Q(s, a) + \alpha \cdot \Big[ R(s, a) + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \Big]
$$

### ❓ Explain all terms of the Q-learning update rule.
- **Q(s, a)**: 
- **α**: 
- **R(s, a)**: 
- **γ**: 
- **s'**: 
- **a'**: 
- **max Q(s', a')**: 

In [ ]:
# Suggestion: you can wrap the code to learn Q-table in a function,
# which you can also reuse in Problem 2

# result of the training is:
#   updated Q table (q_table)
#   total_reward
#   you are free to add others, if you wish

env.verbose = False # disable printouts while training

# ====================================
# YOUR CODE GOES HERE
# ====================================

total_reward = None
explore_rate = None
learning_rate = None

# iterate over number of episodes
    # remember to reset the state!
    # update the exploration and learning rate
    # iterate over number of steps in each Episode
        # Tip: use helper functions defined above to navigate the environment
        # get action and step through the environment
        # update the q-table
        # check if we are done
        
# Tip : print variables of interest, like episode, total_reward, explore_rate
# print(f"Episode: {episode}  Total_reward: {total_reward}  Explore rate: {explore_rate}")
# print("Final Q values: ", q_table)

In [ ]:
print(f"Total reward after training: {total_reward}, total Explore rate after training: {explore_rate:.4f}")

### 🐥 Let's see this in action

In [ ]:
# now we show one episode
# yellow spot should be able to find black spot here
# If you want to see frame by frame, remember to pass "human" as argument in env.render()!

env.verbose = False # set to True to enable messages
NUM_STEPS = 32

observation = env.reset()
state_id = env.get_state_id()
total_reward = 0

image_samples = []

img = env.render()
for _ in range(NUM_STEPS):
    action = update_action(env, state_id, 0, q_table) # explore rate 0: always pick the best action
    observation, reward, terminated, info = env.step(action)
    state_id = env.get_state_id()
    total_reward += reward
    img = env.render()
    image_samples.append(img)
    if terminated:
        break

print(f"Total_reward: ", total_reward)

In [ ]:
# let's make an animation of this, because it's fun
# nothing for you to do here - just run the code
# and open the animation.gif in output/

import os
import matplotlib.animation as animation

height = img.shape[0]
width = img.shape[1]

output_folder = 'output'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)


fig = plt.figure()
imgs = []
for i in range(len(image_samples)):
    im = plt.imshow(image_samples[i].reshape(height, width, 3), animated=True)
    imgs.append([im])

animate = animation.ArtistAnimation(fig, imgs, interval=80, blit=True, repeat_delay=1000)
filename_gif = os.path.join(output_folder, 'animation.gif')
animate.save(filename_gif)
plt.show()

In [ ]:
# Let's visualize the Q values now!
def show_q_heatmap(env, q_table, action: int = None):
    # ====================================
    # YOUR CODE GOES HERE
    # ====================================
                
    # ====================================
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(q_grid, cmap='RdYlGn', interpolation='nearest')
    fig.colorbar(im, ax=ax, label='Max Q-value' if action is None else f'Q-value (action {action})')
    
    for r in range(nrows):
        for c in range(ncols):
            if not np.isnan(q_grid[r, c]):
                ax.text(c, r, f'{q_grid[r, c]:.1f}', ha='center', va='center',
                        fontsize=7, color='black')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Q-Value Heatmap " + (f"(action {action})" if action is not None else "(max over actions)"))
    fig.tight_layout()
    plt.show()

show_q_heatmap(env, q_table)
show_q_heatmap(env, q_table, 0)
show_q_heatmap(env, q_table, 1)
show_q_heatmap(env, q_table, 2)
show_q_heatmap(env, q_table, 3)

In [ ]:
print(q_table.shape)
print(f"Obs space dimensions :{env.observation_space.shape}")

In [ ]:
q_table.argmax(axis=1).reshape(env.observation_space.shape)

In [ ]:
env.close()

### ❓ What do the heatmaps show?

### ❓ What happens with γ = 0 or γ = 1?

### ❓ Every step gives reward 0. How does the agent still learn the shortest path?

### ❓ What happens if the exploration rate is 0 from the start?

### ❓ Why do we set the exploration rate to 0 when we show the trained agent?

## 🌉 Problem 2: Q-learning in the Trondheim environment
### ❗️ The problem
In the second problem, you will train an agent to navigate from a start position to a goal position using Q-learning, similar to the first problem, but now in a larger and more complex map of Trondheim with water (not allowed), land, bus lanes and bus stops (which teleport the agent to the other bus stop).

### 📋 Formal requirements
1. **Implementation**:
   - Initialize the Q-table
   - Reuse the helper functions for Q-learning from Problem 1
   - Choose and tune the hyperparameters
   - Train the agent using Q-learning
   - Plot the number of steps per episode during training
   - Run one episode with the trained agent and verify that it reaches the goal
2. **Discussion**:
   - Has the training converged?
   - Does the agent use the bus stops? Why or why not?
   - What changes with `dead_allowed=True` or `start_random=True`?

In [ ]:
env_config = {
    'env_image_path':'image_TRD.png',
    'dead_allowed':False,
    'start_random':False,
}

if not os.path.exists(env_config['env_image_path']):
  print(f"Error: The image path {env_config['env_image_path']} does not exist.")
else:
  env = TrondheimEnv(conf=env_config)
  print(f"Environment successfully created")

In [ ]:
env.reset()
_ = env.render('human')

In [ ]:
# select the action here
my_action = 'right'

# here is your action
obs, rew, term, info = env.step(action_dict[my_action])
print(rew)
_ = env.render('human')

### Step 1: Initialize Q table

In [ ]:
# ====================================
# YOUR CODE GOES HERE
# ====================================

### Step 2: Write necessary helper functions for Q-learning

In [ ]:
# Task 2: Write necessary helper functions for Q-learning
# ====================================
# YOUR CODE GOES HERE
# ====================================

# YOU DON'T NEED TO DO ANYTHING HERE. The helper functions from Problem 1 can be reused

### Step 3: Play with Hyperparameters

In [ ]:
# ====================================
# YOUR CODE GOES HERE
# ====================================
GAMMA = None
NUM_EPISODES = None
MIN_EXPLORE_RATE = None
MIN_LEARNING_RATE = None
STEP_SIZE = None

### 🐤 Now use Q-learning to learn the Q-table
### Step 4: Train the agent using Q-learning

In [ ]:
env.verbose = False # disable printouts while training

# ====================================
# YOUR CODE GOES HERE
# ====================================

total_reward = None
steps_per_episode = None # number of steps the agent needed in each episode
explore_rate = None
learning_rate = None

In [ ]:
print(f"Total reward after training: {total_reward}, total Explore rate after training: {explore_rate:.4f}")

In [ ]:
# plot the number of steps the agent needed in each episode
plt.figure(figsize=(8, 4))
plt.plot(steps_per_episode, alpha=0.4, label='Steps per episode')
plt.plot(np.convolve(steps_per_episode, np.ones(50) / 50, mode='valid'), label='Moving average (50 episodes)')
plt.yscale('log')
plt.xlabel('Episode')
plt.ylabel('Steps')
plt.title('Steps per episode during training')
plt.legend()
plt.show()

### ❓ Has the training converged?

### 🐥 Let's see one episode

In [ ]:
env.verbose = False
STEP_SIZE = 100

image_samples = []  # Array to store images

observation = env.reset()
state_id = env.get_state_id()
total_reward = 0

img = env.render()
for _ in range(STEP_SIZE):
    action = update_action(env, state_id, 0, q_table) # explore rate 0: always pick the best action
    observation, reward, terminated, info = env.step(action)
    state_id = env.get_state_id()
    total_reward += reward
    img = env.render()
    image_samples.append(img)
    if terminated:
        break

print(f"Total_reward: ", total_reward)

### Simple animation of agent movement in one episode

In [ ]:
# let's make an animation of this, because it's fun
# nothing for you to do here - just run the code

import os
import matplotlib.animation as animation

height = img.shape[0]
width = img.shape[1]

output_folder = 'output'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

fig = plt.figure()
imgs = []
for i in range(len(image_samples)):
    im = plt.imshow(image_samples[i].reshape(height, width, 3), animated=True)
    imgs.append([im])

animate = animation.ArtistAnimation(fig, imgs, interval=80, blit=True, repeat_delay=1000)
filename_gif = os.path.join(output_folder, 'animation_trd.gif')
animate.save(filename_gif)
plt.show()

### ❓ Does the agent use the bus stops? Why or why not?

### ❓ What changes with `dead_allowed=True` or `start_random=True`?

In [ ]:
# ====================================
# YOUR CODE GOES HERE: plot the q values!
# ====================================